# Crear un despliegue

Vamos a crear un despliegue de la aplicación `task_maistro` que construimos en el módulo 5.

## Estructura del código

Para crear un despliegue de LangGraph Platform [hay que entregar la siguiente información](https://docs.langchain.com/langsmith/application-structure):

* Un [archivo de configuración de la API de LangGraph](https://docs.langchain.com/langsmith/application-structure#configuration-file): `langgraph.json`
* Los grafos que implementan la lógica de la aplicación, por ejemplo `task_maistro.py`
* Un archivo que especifica las dependencias necesarias para ejecutar la aplicación: `requirements.txt`
* Las variables de entorno que la aplicación necesita para funcionar: `.env` o `docker-compose.yml`

¡Todo esto ya lo tenemos en el directorio `module-6/deployment`!

## CLI

La [CLI de LangGraph](https://docs.langchain.com/langsmith/cli) es una interfaz de línea de comandos para crear un despliegue de LangGraph Platform.

In [1]:
%%capture --no-stderr
%pip install -U langgraph-cli

Para crear un [despliegue autoalojado](https://docs.langchain.com/langsmith/self_hosted_data_plane) (*self-hosted*), seguiremos unos pocos pasos.

### Construir la imagen de Docker del servidor de LangGraph

Primero usamos la CLI de langgraph para crear una imagen de Docker del [servidor de LangGraph](https://docs.google.com/presentation/d/18MwIaNR2m4Oba6roK_2VQcBE_8Jq_SI7VHTXJdl7raU/edit#slide=id.g313fb160676_0_32).

Esto empaquetará nuestro grafo y sus dependencias en una imagen de Docker.

Una imagen de Docker es una plantilla para un contenedor de Docker que contiene el código y las dependencias necesarias para ejecutar la aplicación.

Asegúrate de tener [Docker](https://docs.docker.com/engine/install/) instalado y ejecuta el siguiente comando para crear la imagen de Docker, `my-image`:

```
$ cd module-6/deployment
$ langgraph build -t my-image
```

### Configurar Redis y PostgreSQL

Si ya tienes Redis y PostgreSQL funcionando (por ejemplo, en tu equipo o en otros servidores), crea y ejecuta el contenedor del servidor de LangGraph [por separado](https://docs.langchain.com/langsmith/deploy-hybrid#running-the-application-locally) con las URI de Redis y de PostgreSQL:

```
docker run \
    --env-file .env \
    -p 8123:8000 \
    -e REDIS_URI="foo" \
    -e DATABASE_URI="bar" \
    -e LANGSMITH_API_KEY="baz" \
    my-image
```

Como alternativa, puedes usar el archivo `docker-compose.yml` incluido para crear tres contenedores separados a partir de los servicios definidos:

* `langgraph-redis`: crea un contenedor nuevo con la imagen oficial de Redis.
* `langgraph-postgres`: crea un contenedor nuevo con la imagen oficial de Postgres.
* `langgraph-api`: crea un contenedor nuevo con tu imagen ya construida.

Simplemente copia `docker-compose-example.yml` y agrega las siguientes variables de entorno para ejecutar la aplicación `task_maistro` desplegada:

* `IMAGE_NAME` (por ejemplo, `my-image`)
* `LANGSMITH_API_KEY`
* `OPENAI_API_KEY`

Luego, ¡lanza el despliegue!

```
$ cd module-6/deployment
$ docker compose up
```